# ACANet Baseline: Basketball Shot Classification (3pt / 2pt / Free Throw)
### Action-Context Attention Network — Basketball-51

This notebook implements **ACANet (Action-Context Attention Network)** for video-based shot classification.

**Core idea — the Action-Context relationship:**
> In basketball, a shot's class depends on *two complementary signals*:
> 1. **Action** — the player's shooting motion, ball trajectory, and body pose (temporal dynamics).
> 2. **Context** — the court layout, distance to the hoop, and the 3-point line (spatial scene understanding).
>
> ACANet uses a **dual-branch architecture** where the Action branch captures *what is happening* and the Context branch captures *where it is happening*. An **attention mechanism** then lets the Action features query the Context features, forcing the model to attend to court regions that are diagnostically relevant (e.g., the 3-point arc).

**Technical specs:**
- Dual-branch 3D ResNet backbone with Action-Context Attention
- Label Smoothing Cross-Entropy, AdamW + OneCycleLR
- 16-frame clips (stride 2), `decord` video loading
- Attention heatmap visualisation for failure-mode analysis

## 1. Setup & Dependencies

In [ ]:
# ── Install dependencies ──
!pip install -q decord einops fvcore av \
    scikit-learn seaborn matplotlib tqdm opencv-python-headless

# ── Mount Google Drive ──
from google.colab import drive
drive.mount('/content/drive')

import os, sys, math, time, random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as T
import warnings
warnings.filterwarnings('ignore')

# ── Dataset root on Drive ──
DATASET_ROOT = '/content/drive/MyDrive/Basketball-51'
SAVE_DIR     = '/content/drive/MyDrive/Basketball-51/checkpoints_acanet'
os.makedirs(SAVE_DIR, exist_ok=True)

assert os.path.isdir(DATASET_ROOT), f"Dataset not found at {DATASET_ROOT}"

# List classes
class_dirs = sorted([
    d for d in os.listdir(DATASET_ROOT)
    if os.path.isdir(os.path.join(DATASET_ROOT, d)) and d != 'checkpoints_acanet'
])
print(f"Found {len(class_dirs)} class folders: {class_dirs}")
for d in class_dirs:
    count = len([f for f in os.listdir(os.path.join(DATASET_ROOT, d))
                 if os.path.isfile(os.path.join(DATASET_ROOT, d, f))])
    print(f"  {d}: {count} videos")

## 2. GPU Configuration & Hyperparameters

In [ ]:
# ── Reproducibility ──
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = True

# ── Device ──
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("⚠ No GPU — training will be very slow!")

# ── Hyperparameters ──
CFG = {
    # Data
    'NUM_FRAMES':     16,
    'FRAME_STRIDE':    2,       # sample every 2nd frame → covers 32 raw frames
    'FRAME_SIZE':    224,
    'NUM_CLASSES':     3,
    'BATCH_SIZE':      6,
    'NUM_WORKERS':     4,
    'PIN_MEMORY':   True,

    # Training
    'EPOCHS':         50,
    'LR':           3e-4,
    'WEIGHT_DECAY': 1e-2,
    'LABEL_SMOOTH':  0.1,       # label smoothing factor

    # Architecture
    'EMBED_DIM':    512,        # feature dim per branch
    'ATTN_HEADS':     4,        # multi-head attention heads
    'DROPOUT':      0.4,
}

CLASS_NAMES  = ['2pt', '3pt', 'FreeThrow']
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}

print("\nConfiguration:")
for k, v in CFG.items():
    print(f"  {k}: {v}")
print(f"\nClasses: {CLASS_TO_IDX}")

## 3. Data Pipeline — `Basketball51Dataset` (decord)

Uses **decord** for GPU-accelerated / zero-copy video decoding.  
**Sampling strategy:** 16 frames with stride 2 → each clip effectively covers 32 raw frames.  
**Augmentation:** Random crop, horizontal flip, colour jitter (training only).

In [ ]:
from decord import VideoReader, cpu as decord_cpu

# ImageNet stats
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]


def get_transforms(is_train: bool):
    """Spatial transforms applied per-frame."""
    if is_train:
        return T.Compose([
            T.ToPILImage(),
            T.RandomResizedCrop(CFG['FRAME_SIZE'], scale=(0.8, 1.0)),
            T.RandomHorizontalFlip(p=0.5),
            T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
            T.ToTensor(),
            T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])
    else:
        return T.Compose([
            T.ToPILImage(),
            T.Resize(int(CFG['FRAME_SIZE'] * 1.14)),   # 256
            T.CenterCrop(CFG['FRAME_SIZE']),
            T.ToTensor(),
            T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])


class Basketball51Dataset(Dataset):
    """
    Video dataset using decord for efficient frame loading.
    Returns clips of shape (C, T, H, W).

    Sampling: pick 16 frames with stride 2, giving temporal coverage
    of 32 raw frames per clip. Temporal jittering is added during
    training by randomly offsetting the start position.
    """

    VIDEO_EXTS = {'.mp4', '.avi', '.mov', '.mkv', '.webm'}

    def __init__(self, video_paths, labels, is_train=True):
        self.video_paths = video_paths
        self.labels = labels
        self.is_train = is_train
        self.num_frames = CFG['NUM_FRAMES']
        self.stride = CFG['FRAME_STRIDE']
        self.transform = get_transforms(is_train)

    def __len__(self):
        return len(self.video_paths)

    def _sample_frame_indices(self, total_frames):
        """
        Compute frame indices for 16 frames with stride 2.
        Training: random start offset (temporal jittering).
        Validation: centre-aligned sampling.
        """
        clip_len = self.num_frames * self.stride  # 32 raw frames needed

        if total_frames >= clip_len:
            max_start = total_frames - clip_len
            start = random.randint(0, max_start) if self.is_train else max_start // 2
        else:
            start = 0

        indices = [start + i * self.stride for i in range(self.num_frames)]
        # Clamp to valid range (handles short videos)
        indices = [min(idx, total_frames - 1) for idx in indices]
        return indices

    def __getitem__(self, idx):
        path = self.video_paths[idx]
        label = self.labels[idx]

        try:
            vr = VideoReader(path, ctx=decord_cpu(0))
        except Exception as e:
            print(f"[ERROR] Cannot read {path}: {e}")
            # Return a zero tensor as fallback
            clip = torch.zeros(3, self.num_frames, CFG['FRAME_SIZE'], CFG['FRAME_SIZE'])
            return clip, label

        total = len(vr)
        indices = self._sample_frame_indices(total)
        frames = vr.get_batch(indices).asnumpy()  # (T, H, W, C) uint8

        # Apply spatial transforms per frame → list of (C, H, W) tensors
        frames_t = [self.transform(frames[t]) for t in range(frames.shape[0])]

        # Stack → (C, T, H, W)
        clip = torch.stack(frames_t, dim=1)
        return clip, label


# ── Helper: scan dataset directory ──
def scan_dataset(root, class_to_idx):
    paths, labels = [], []
    for cls_name, idx in class_to_idx.items():
        cls_dir = os.path.join(root, cls_name)
        if not os.path.isdir(cls_dir):
            print(f"  [WARN] Missing folder: {cls_dir}")
            continue
        for f in sorted(os.listdir(cls_dir)):
            if os.path.splitext(f)[1].lower() in Basketball51Dataset.VIDEO_EXTS:
                paths.append(os.path.join(cls_dir, f))
                labels.append(idx)
    return paths, labels


print("Basketball51Dataset (decord) defined ✓")

## 4. Data Loaders (80 / 20 Stratified Split)

In [ ]:
from sklearn.model_selection import train_test_split
from collections import Counter

all_paths, all_labels = scan_dataset(DATASET_ROOT, CLASS_TO_IDX)
print(f"Total videos: {len(all_paths)}")
print("Distribution:", {CLASS_NAMES[k]: v for k, v in sorted(Counter(all_labels).items())})

train_paths, val_paths, train_labels, val_labels = train_test_split(
    all_paths, all_labels,
    test_size=0.20, stratify=all_labels, random_state=SEED,
)

print(f"\nTrain: {len(train_paths)}  |  Val: {len(val_paths)}")
print("Train:", {CLASS_NAMES[k]: v for k, v in sorted(Counter(train_labels).items())})
print("Val:  ", {CLASS_NAMES[k]: v for k, v in sorted(Counter(val_labels).items())})

train_ds = Basketball51Dataset(train_paths, train_labels, is_train=True)
val_ds   = Basketball51Dataset(val_paths,   val_labels,   is_train=False)

train_loader = DataLoader(train_ds, batch_size=CFG['BATCH_SIZE'], shuffle=True,
                          num_workers=CFG['NUM_WORKERS'], pin_memory=CFG['PIN_MEMORY'],
                          drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=CFG['BATCH_SIZE'], shuffle=False,
                          num_workers=CFG['NUM_WORKERS'], pin_memory=CFG['PIN_MEMORY'])

print(f"\nBatches — train: {len(train_loader)}, val: {len(val_loader)}")

# Sanity check
clip, lbl = next(iter(train_loader))
print(f"Sample batch: clip {clip.shape}, labels {lbl}")

## 5. ACANet Architecture

### Design Overview

```
                    ┌─────────────────┐
  Video clip ──────►│  Shared 3D Stem  │
  (B,3,16,224,224)  └────────┬────────┘
                             │
                ┌────────────┴────────────┐
                ▼                         ▼
        ┌──────────────┐         ┌──────────────┐
        │ Action Branch│         │Context Branch │
        │  (temporal)  │         │  (spatial)    │
        │  3D ResBlocks│         │  3D ResBlocks │
        │  ──► GAP_t   │         │  ──► GAP_s    │
        └──────┬───────┘         └──────┬────────┘
               │ Q (query)              │ K, V (keys/values)
               └──────┐    ┌───────────┘
                       ▼    ▼
               ┌───────────────────┐
               │ Action-Context    │
               │ Multi-Head Attn   │
               └────────┬──────────┘
                        │
                        ▼
               ┌───────────────────┐
               │ Classification    │
               │ Head (MLP → 3)    │
               └───────────────────┘
```

**Action Branch** pools spatially to produce a *temporal descriptor* — *what motion is occurring?*  
**Context Branch** pools temporally to produce a *spatial descriptor* — *where on the court?*  
The **ACA attention** lets action features attend to context features, weighting court regions by their relevance to the detected motion pattern.

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  5a. 3D ResNet Building Blocks (shared between branches)
# ═══════════════════════════════════════════════════════════════

class BasicBlock3D(nn.Module):
    """Residual block with two 3×3×3 convolutions."""
    expansion = 1

    def __init__(self, in_ch, out_ch, stride=1, temporal_stride=1, downsample=None):
        super().__init__()
        self.conv1 = nn.Conv3d(in_ch, out_ch, 3,
                               stride=(temporal_stride, stride, stride),
                               padding=1, bias=False)
        self.bn1   = nn.BatchNorm3d(out_ch)
        self.conv2 = nn.Conv3d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm3d(out_ch)
        self.relu  = nn.ReLU(inplace=True)
        self.downsample = downsample

    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.downsample is not None:
            identity = self.downsample(x)
        return self.relu(out + identity)


class Bottleneck3D(nn.Module):
    """Bottleneck block: 1×1×1 → 3×3×3 → 1×1×1."""
    expansion = 4

    def __init__(self, in_ch, mid_ch, stride=1, temporal_stride=1, downsample=None):
        super().__init__()
        out_ch = mid_ch * self.expansion
        self.conv1 = nn.Conv3d(in_ch, mid_ch, 1, bias=False)
        self.bn1   = nn.BatchNorm3d(mid_ch)
        self.conv2 = nn.Conv3d(mid_ch, mid_ch, 3,
                               stride=(temporal_stride, stride, stride),
                               padding=1, bias=False)
        self.bn2   = nn.BatchNorm3d(mid_ch)
        self.conv3 = nn.Conv3d(mid_ch, out_ch, 1, bias=False)
        self.bn3   = nn.BatchNorm3d(out_ch)
        self.relu  = nn.ReLU(inplace=True)
        self.downsample = downsample

    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        if self.downsample is not None:
            identity = self.downsample(x)
        return self.relu(out + identity)


def _make_layer(block, in_ch, mid_ch, num_blocks, stride=1, temporal_stride=1):
    """Build a sequence of residual blocks."""
    out_ch = mid_ch * block.expansion
    downsample = None
    if stride != 1 or in_ch != out_ch:
        downsample = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 1,
                      stride=(temporal_stride, stride, stride), bias=False),
            nn.BatchNorm3d(out_ch),
        )
    layers = [block(in_ch, mid_ch, stride, temporal_stride, downsample)]
    for _ in range(1, num_blocks):
        layers.append(block(out_ch, mid_ch))
    return nn.Sequential(*layers), out_ch


print("3D ResNet blocks defined ✓")

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  5b. Action-Context Attention (ACA) Module
# ═══════════════════════════════════════════════════════════════

class ActionContextAttention(nn.Module):
    """
    Multi-head attention where:
      - Query  = Action features  (temporal descriptor of player motion)
      - Key/Value = Context features (spatial descriptor of court layout)

    This lets the model *ask*:  "Given this shooting motion, which parts
    of the court (3-pt line, free-throw line, paint area) are most
    relevant for classifying the shot?"

    The attention weights are stored for later visualisation as heatmaps
    over the original video frames.
    """

    def __init__(self, embed_dim: int, num_heads: int = 4, dropout: float = 0.1):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim  = embed_dim // num_heads
        assert embed_dim % num_heads == 0, "embed_dim must be divisible by num_heads"

        # Separate projections for Q (action) and K/V (context)
        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)

        self.scale = self.head_dim ** -0.5
        self.attn_drop = nn.Dropout(dropout)
        self.layer_norm = nn.LayerNorm(embed_dim)

        # Store attention weights for visualisation
        self.attn_weights = None

    def forward(self, action_tokens, context_tokens):
        """
        Args:
            action_tokens:  (B, T_a, D)  — temporal "what is happening" tokens
            context_tokens: (B, S_c, D)  — spatial  "where on the court" tokens
        Returns:
            attended: (B, T_a, D)  — action features enriched by court context
        """
        B, T, D = action_tokens.shape
        S = context_tokens.shape[1]

        # Project
        Q = self.q_proj(action_tokens).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.k_proj(context_tokens).view(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.v_proj(context_tokens).view(B, S, self.num_heads, self.head_dim).transpose(1, 2)

        # Scaled dot-product attention
        attn = (Q @ K.transpose(-2, -1)) * self.scale   # (B, heads, T, S)
        attn = attn.softmax(dim=-1)
        self.attn_weights = attn.detach()  # save for heatmaps
        attn = self.attn_drop(attn)

        out = (attn @ V).transpose(1, 2).reshape(B, T, D)
        out = self.out_proj(out)

        # Residual + LayerNorm
        attended = self.layer_norm(action_tokens + out)
        return attended


print("ActionContextAttention module defined ✓")

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  5c. Full ACANet Model
# ═══════════════════════════════════════════════════════════════

class ACANet(nn.Module):
    """
    Action-Context Attention Network.

    Architecture:
      1. Shared 3D stem (conv + pool)
      2. Two parallel branches:
         - Action Branch  → temporal features (pools spatial dims)
         - Context Branch → spatial features  (pools temporal dim)
      3. Action-Context Attention — action queries attend to context keys
      4. MLP classification head (concatenated features → 3 classes)

    Input:  (B, 3, 16, 224, 224)
    Output: (B, num_classes)
    """

    def __init__(self, num_classes=3, embed_dim=512, attn_heads=4, dropout=0.4):
        super().__init__()
        block = Bottleneck3D

        # ── Shared Stem ──
        self.stem = nn.Sequential(
            nn.Conv3d(3, 64, kernel_size=(3, 7, 7), stride=(1, 2, 2),
                      padding=(1, 3, 3), bias=False),
            nn.BatchNorm3d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool3d(kernel_size=(1, 3, 3), stride=(1, 2, 2), padding=(0, 1, 1)),
        )

        # ── Action Branch (focuses on temporal dynamics / motion) ──
        # Uses temporal downsampling to compress spatial info and
        # preserve fine-grained temporal patterns.
        self.action_layer1, ch1 = _make_layer(block, 64,  64,  3, stride=1, temporal_stride=1)
        self.action_layer2, ch2 = _make_layer(block, ch1, 128, 4, stride=2, temporal_stride=1)
        self.action_layer3, ch3 = _make_layer(block, ch2, 256, 6, stride=2, temporal_stride=2)
        self.action_pool = nn.AdaptiveAvgPool3d((None, 1, 1))  # pool spatial, keep temporal
        self.action_proj = nn.Linear(ch3, embed_dim)

        # ── Context Branch (focuses on spatial court layout) ──
        # Uses spatial downsampling while pooling the temporal dim
        # to capture *where* things are happening on the court.
        self.context_layer1, c1 = _make_layer(block, 64,  64,  3, stride=1, temporal_stride=1)
        self.context_layer2, c2 = _make_layer(block, c1, 128, 4, stride=2, temporal_stride=2)
        self.context_layer3, c3 = _make_layer(block, c2, 256, 6, stride=2, temporal_stride=2)
        self.context_pool = nn.AdaptiveAvgPool3d((1, None, None))  # pool temporal, keep spatial
        self.context_proj = nn.Linear(c3, embed_dim)

        # ── Action-Context Attention ──
        self.aca_attention = ActionContextAttention(
            embed_dim=embed_dim, num_heads=attn_heads, dropout=dropout * 0.5,
        )

        # ── Classification Head ──
        # Concatenates global action descriptor + attended descriptor
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim * 2, embed_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(embed_dim, embed_dim // 2),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout * 0.5),
            nn.Linear(embed_dim // 2, num_classes),
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv3d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm3d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        """
        x: (B, 3, T, H, W) — e.g., (B, 3, 16, 224, 224)
        """
        B = x.shape[0]

        # ── Shared stem ──
        shared = self.stem(x)  # (B, 64, T, 56, 56)

        # ────── Action Branch ──────
        # Goal: produce temporal tokens (B, T', D)
        a = self.action_layer1(shared)
        a = self.action_layer2(a)
        a = self.action_layer3(a)                 # (B, 1024, T', H', W')
        a = self.action_pool(a)                   # (B, 1024, T', 1, 1)
        a = a.squeeze(-1).squeeze(-1).permute(0, 2, 1)  # (B, T', 1024)
        action_tokens = self.action_proj(a)       # (B, T', embed_dim)

        # ────── Context Branch ──────
        # Goal: produce spatial tokens (B, S, D) where S = H'×W'
        c = self.context_layer1(shared)
        c = self.context_layer2(c)
        c = self.context_layer3(c)                # (B, 1024, T', H', W')
        c = self.context_pool(c)                  # (B, 1024, 1, H', W')
        c = c.squeeze(2)                          # (B, 1024, H', W')
        B2, C2, H2, W2 = c.shape
        c = c.reshape(B2, C2, H2 * W2).permute(0, 2, 1)  # (B, S, 1024)
        context_tokens = self.context_proj(c)     # (B, S, embed_dim)

        # ────── Action-Context Attention ──────
        # Action queries attend to context keys/values
        attended = self.aca_attention(action_tokens, context_tokens)  # (B, T', D)

        # ────── Classification ──────
        # Global action descriptor (mean-pool temporal tokens)
        action_global = action_tokens.mean(dim=1)   # (B, D)
        # Attended descriptor (mean-pool attended tokens)
        attended_global = attended.mean(dim=1)       # (B, D)

        # Concatenate and classify
        fused = torch.cat([action_global, attended_global], dim=1)  # (B, 2D)
        logits = self.classifier(fused)              # (B, num_classes)
        return logits


print("ACANet defined ✓")

## 6. Model Instantiation & GPU Transfer

In [ ]:
model = ACANet(
    num_classes=CFG['NUM_CLASSES'],
    embed_dim=CFG['EMBED_DIM'],
    attn_heads=CFG['ATTN_HEADS'],
    dropout=CFG['DROPOUT'],
).to(device)

# ── Parameter count ──
total_p = sum(p.numel() for p in model.parameters())
train_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"ACANet")
print(f"  Total params:     {total_p:,}")
print(f"  Trainable params: {train_p:,}")

# ── Sanity check ──
dummy = torch.randn(2, 3, CFG['NUM_FRAMES'], CFG['FRAME_SIZE'], CFG['FRAME_SIZE']).to(device)
with torch.no_grad():
    out = model(dummy)
print(f"  Forward: {dummy.shape} → {out.shape}")
assert out.shape == (2, CFG['NUM_CLASSES'])

# Check attention weights were stored
attn_w = model.aca_attention.attn_weights
print(f"  Attention weights shape: {attn_w.shape}")  # (B, heads, T', S)

del dummy, out
torch.cuda.empty_cache()
print("  Sanity check passed ✓")

## 7. Loss, Optimizer & Scheduler

- **Label Smoothing CE** (`smoothing=0.1`) — softens one-hot targets to improve generalisation  
- **AdamW** — decoupled weight decay for better regularisation  
- **OneCycleLR** — cosine-annealing with linear warm-up for faster convergence

In [ ]:
# Label Smoothing Cross-Entropy
criterion = nn.CrossEntropyLoss(label_smoothing=CFG['LABEL_SMOOTH']).to(device)

# AdamW optimizer
optimizer = optim.AdamW(
    model.parameters(),
    lr=CFG['LR'],
    weight_decay=CFG['WEIGHT_DECAY'],
)

# OneCycleLR — total steps = epochs × batches_per_epoch
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=CFG['LR'],
    epochs=CFG['EPOCHS'],
    steps_per_epoch=len(train_loader),
    pct_start=0.1,         # 10% warm-up
    anneal_strategy='cos',
)

# Mixed-precision scaler
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

print("Label-smoothing CE, AdamW, OneCycleLR, AMP scaler configured ✓")

## 8. Training & Validation Loops

In [ ]:
from tqdm.auto import tqdm
from sklearn.metrics import f1_score


def train_one_epoch(model, loader, criterion, optimizer, scheduler, scaler, device):
    """Train for one epoch. Returns (avg_loss, accuracy %)."""
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    pbar = tqdm(loader, desc='  Train', leave=False)
    for clips, labels in pbar:
        clips  = clips.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
            logits = model(clips)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()  # OneCycleLR steps per batch, not per epoch

        running_loss += loss.item() * clips.size(0)
        _, preds = logits.max(1)
        correct += preds.eq(labels).sum().item()
        total   += labels.size(0)

        pbar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{100.*correct/total:.1f}%")

    return running_loss / total, 100. * correct / total


@torch.no_grad()
def validate(model, loader, criterion, device):
    """
    Validate. Returns (avg_loss, accuracy %, weighted_f1, all_preds, all_targets).
    """
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_targets = [], []

    pbar = tqdm(loader, desc='  Val  ', leave=False)
    for clips, labels in pbar:
        clips  = clips.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
            logits = model(clips)
            loss = criterion(logits, labels)

        running_loss += loss.item() * clips.size(0)
        _, preds = logits.max(1)
        correct += preds.eq(labels).sum().item()
        total   += labels.size(0)

        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(labels.cpu().numpy())

        pbar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{100.*correct/total:.1f}%")

    avg_loss = running_loss / total
    acc = 100. * correct / total
    wf1 = f1_score(all_targets, all_preds, average='weighted') * 100.
    return avg_loss, acc, wf1, all_preds, all_targets


print("train_one_epoch() and validate() defined ✓")

## 9. Execute Training (50 Epochs)

Best checkpoint saved by validation accuracy. Logs accuracy **and** weighted F1 per epoch.

In [ ]:
BEST_PATH = os.path.join(SAVE_DIR, 'best_acanet.pth')

history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': [], 'val_f1': [],
    'lr': [],
}

best_val_acc = 0.0
patience_counter = 0
PATIENCE = 15

print(f"Training ACANet for {CFG['EPOCHS']} epochs …\n")
t0 = time.time()

for epoch in range(1, CFG['EPOCHS'] + 1):
    ep_start = time.time()

    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, scheduler, scaler, device,
    )
    val_loss, val_acc, val_f1, val_preds, val_targets = validate(
        model, val_loader, criterion, device,
    )

    current_lr = optimizer.param_groups[0]['lr']
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)
    history['lr'].append(current_lr)

    elapsed = time.time() - ep_start
    print(f"Epoch {epoch:>3}/{CFG['EPOCHS']}  │  "
          f"Train  L:{train_loss:.4f} A:{train_acc:5.1f}%  │  "
          f"Val  L:{val_loss:.4f} A:{val_acc:5.1f}% F1:{val_f1:5.1f}%  │  "
          f"LR:{current_lr:.2e}  │  {elapsed:.0f}s")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'val_acc': val_acc, 'val_f1': val_f1,
            'cfg': CFG,
        }, BEST_PATH)
        print(f"  ↳ Best model saved (acc={val_acc:.1f}%, f1={val_f1:.1f}%)")
    else:
        patience_counter += 1

    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch} (no improvement for {PATIENCE} epochs).")
        break

total_min = (time.time() - t0) / 60
print(f"\n{'='*65}")
print(f"Done in {total_min:.1f} min  |  Best val accuracy: {best_val_acc:.1f}%")

## 10. Training Curves

In [ ]:
import matplotlib.pyplot as plt

epochs_range = range(1, len(history['train_loss']) + 1)
best_ep = int(np.argmax(history['val_acc'])) + 1

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(epochs_range, history['train_loss'], label='Train')
axes[0].plot(epochs_range, history['val_loss'],   label='Val')
axes[0].axvline(best_ep, color='gray', ls='--', alpha=0.5, label=f'Best ({best_ep})')
axes[0].set(xlabel='Epoch', ylabel='Loss', title='Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Accuracy
axes[1].plot(epochs_range, history['train_acc'], label='Train')
axes[1].plot(epochs_range, history['val_acc'],   label='Val')
axes[1].axvline(best_ep, color='gray', ls='--', alpha=0.5)
axes[1].set(xlabel='Epoch', ylabel='Accuracy (%)', title='Accuracy')
axes[1].legend(); axes[1].grid(alpha=0.3)

# Weighted F1
axes[2].plot(epochs_range, history['val_f1'], label='Val Weighted F1', color='tab:green')
axes[2].axvline(best_ep, color='gray', ls='--', alpha=0.5)
axes[2].set(xlabel='Epoch', ylabel='F1 (%)', title='Weighted F1-Score')
axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, 'acanet_curves.png'), dpi=150)
plt.show()

## 11. Confusion Matrix & Classification Report

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# ── Load best model ──
ckpt = torch.load(BEST_PATH, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
print(f"Loaded best ACANet from epoch {ckpt['epoch']} "
      f"(acc={ckpt['val_acc']:.1f}%, f1={ckpt['val_f1']:.1f}%)\n")

# ── Final validation pass ──
_, final_acc, final_f1, all_preds, all_targets = validate(
    model, val_loader, criterion, device)
print(f"Top-1 Accuracy: {final_acc:.1f}%  |  Weighted F1: {final_f1:.1f}%\n")

# ── Confusion Matrix ──
cm = confusion_matrix(all_targets, all_preds)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            ax=ax, linewidths=0.5)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j + 0.5, i + 0.72, f"({cm_pct[i,j]:.0f}%)",
                ha='center', va='center', fontsize=9, color='gray')
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('True', fontsize=12)
ax.set_title(f'ACANet Confusion Matrix (Acc: {final_acc:.1f}%)', fontsize=13)
plt.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, 'acanet_confusion_matrix.png'), dpi=150)
plt.show()

# ── Classification Report ──
print("\nClassification Report:\n")
print(classification_report(all_targets, all_preds,
                            target_names=CLASS_NAMES, digits=3))

## 12. Attention Heatmap Visualisation

This is the key diagnostic tool for ACANet. The attention weights from the
ACA module tell us **which spatial regions of the court the Action branch
is attending to** when making its decision.

If the model is working correctly, we expect:
- **3pt shots:** high attention on the 3-point arc region
- **2pt shots:** attention focused on the paint / mid-range area
- **Free throws:** attention focused on the free-throw line

If the model is *failing*, the heatmaps may reveal that it is being
distracted by **crowd noise, lighting artefacts, or scoreboard text**.

In [ ]:
import cv2


def generate_attention_heatmaps(model, video_path, device, num_display=4):
    """
    Run a single video through ACANet, extract the Action-Context
    attention weights, and overlay them as heatmaps on the original frames.

    The attention weights have shape (1, heads, T_action, S_context)
    where S_context = H' × W' is the flattened spatial grid from the
    Context branch. We average over heads and temporal queries, then
    reshape back to (H', W') and upsample to overlay on the original frame.

    Args:
        model:  trained ACANet (already on device, in eval mode)
        video_path: path to a single video file
        device: cuda / cpu
        num_display: how many frames to show
    """
    model.eval()

    # ── Load video clip with decord ──
    vr = VideoReader(video_path, ctx=decord_cpu(0))
    total = len(vr)
    stride = CFG['FRAME_STRIDE']
    nf = CFG['NUM_FRAMES']
    clip_len = nf * stride
    start = max(0, (total - clip_len) // 2)
    indices = [min(start + i * stride, total - 1) for i in range(nf)]
    raw_frames = vr.get_batch(indices).asnumpy()  # (T, H, W, 3) uint8

    # ── Preprocess for model ──
    val_tf = get_transforms(is_train=False)
    frames_t = [val_tf(raw_frames[t]) for t in range(nf)]
    clip = torch.stack(frames_t, dim=1).unsqueeze(0).to(device)  # (1,3,T,224,224)

    # ── Forward pass ──
    with torch.no_grad(), torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
        logits = model(clip)

    pred_idx = logits.argmax(dim=1).item()
    confidence = F.softmax(logits, dim=1).max().item()
    pred_label = CLASS_NAMES[pred_idx]

    # ── Extract attention weights ──
    # Shape: (1, num_heads, T_action, S_context)
    attn = model.aca_attention.attn_weights  # stored during forward
    # Average over heads and action-temporal queries → (S_context,)
    attn_map = attn.mean(dim=1).mean(dim=1).squeeze(0)  # (S,)

    # Determine spatial grid size from context branch
    # Run context branch to get H', W'
    with torch.no_grad():
        shared = model.stem(clip)
        c = model.context_layer1(shared)
        c = model.context_layer2(c)
        c = model.context_layer3(c)
        c = model.context_pool(c)  # (1, C, 1, H', W')
        ctx_h, ctx_w = c.shape[-2], c.shape[-1]

    attn_2d = attn_map.cpu().numpy().reshape(ctx_h, ctx_w)

    # ── Visualise ──
    # Select evenly spaced frames to display
    display_indices = np.linspace(0, nf - 1, num_display, dtype=int)

    fig, axes = plt.subplots(2, num_display, figsize=(4 * num_display, 8))

    for i, fi in enumerate(display_indices):
        frame = raw_frames[fi]  # (H, W, 3)
        h_orig, w_orig = frame.shape[:2]

        # Original frame
        axes[0, i].imshow(frame)
        axes[0, i].set_title(f"Frame {indices[fi]}", fontsize=10)
        axes[0, i].axis('off')

        # Upsample attention to original resolution
        attn_upsampled = cv2.resize(attn_2d, (w_orig, h_orig),
                                     interpolation=cv2.INTER_CUBIC)
        # Normalise to [0, 1]
        attn_upsampled = (attn_upsampled - attn_upsampled.min()) / \
                         (attn_upsampled.max() - attn_upsampled.min() + 1e-8)

        # Overlay heatmap
        axes[1, i].imshow(frame)
        axes[1, i].imshow(attn_upsampled, cmap='jet', alpha=0.5)
        axes[1, i].set_title(f"Attention", fontsize=10)
        axes[1, i].axis('off')

    fig.suptitle(f"Prediction: {pred_label}  |  Confidence: {confidence:.1%}",
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    fig.savefig(os.path.join(SAVE_DIR, 'attention_heatmap.png'),
                dpi=150, bbox_inches='tight')
    plt.show()

    print(f"\nPrediction: {pred_label} ({confidence:.1%})")
    print(f"Attention grid: {ctx_h}×{ctx_w} → upsampled to {h_orig}×{w_orig}")
    return pred_label, confidence


print("generate_attention_heatmaps() defined ✓")

In [ ]:
# ── Run attention heatmap on a sample validation video ──
# Pick the first video from each class for illustration
print("Generating attention heatmaps for sample videos …\n")

for cls_name in CLASS_NAMES:
    cls_dir = os.path.join(DATASET_ROOT, cls_name)
    sample_video = sorted(os.listdir(cls_dir))[0]
    sample_path = os.path.join(cls_dir, sample_video)
    print(f"─── {cls_name}: {sample_video} ───")
    generate_attention_heatmaps(model, sample_path, device, num_display=4)
    print()

## 13. Single-Video Inference

Load any video from a local path (e.g., school court footage) and predict the shot type with a confidence score. Useful for testing domain generalisation.

In [ ]:
def predict_single_video(model, video_path, device):
    """
    Run inference on a single video file and return the predicted
    class name and per-class confidence scores.

    Args:
        model: trained ACANet on device
        video_path: path to a .mp4 / .avi video
        device: cuda / cpu

    Returns:
        pred_class (str), confidences (dict)
    """
    model.eval()

    # Load & sample frames
    vr = VideoReader(video_path, ctx=decord_cpu(0))
    total = len(vr)
    stride = CFG['FRAME_STRIDE']
    nf = CFG['NUM_FRAMES']
    clip_len = nf * stride
    start = max(0, (total - clip_len) // 2)
    indices = [min(start + i * stride, total - 1) for i in range(nf)]
    frames = vr.get_batch(indices).asnumpy()

    # Preprocess
    val_tf = get_transforms(is_train=False)
    frames_t = [val_tf(frames[t]) for t in range(nf)]
    clip = torch.stack(frames_t, dim=1).unsqueeze(0).to(device)

    # Inference
    with torch.no_grad(), torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
        logits = model(clip)

    probs = F.softmax(logits, dim=1).squeeze(0).cpu().numpy()
    pred_idx = probs.argmax()
    pred_class = CLASS_NAMES[pred_idx]

    confidences = {CLASS_NAMES[i]: f"{probs[i]:.1%}" for i in range(len(CLASS_NAMES))}

    print(f"Video:      {os.path.basename(video_path)}")
    print(f"Prediction: {pred_class}")
    print(f"Confidence: {confidences}")
    return pred_class, confidences


# ═══ Example usage ═══
# Change this path to test your own school court videos:
TEST_VIDEO = os.path.join(DATASET_ROOT, CLASS_NAMES[0],
                          sorted(os.listdir(os.path.join(DATASET_ROOT, CLASS_NAMES[0])))[0])

print("─── Single-Video Inference ───")
predict_single_video(model, TEST_VIDEO, device)